# voice-clone speed test on a Kaggle GPU

Runs `python -m voice_clone bench` on an NVIDIA card, in fp32 and fp16, so the results can be compared with other machines. No microphone is needed: the test uses made-up speech.

## Before you run

1. **Settings → Accelerator:** choose **GPU P100** (an older card, like a GeForce 10: the app keeps it at fp32). Run the notebook again with **GPU T4 x2** (a newer card: the app uses fp16 there).
2. **Settings → Internet:** turn it **on** (it downloads PyTorch, about 2.5 GB, and the base models, about 560 MB).
3. **Add your voice:** make a **private** dataset with the files from `voices/My-voice/` (the `.pth` and `.index`), then **Add Input** → your dataset. Any folder under `/kaggle/input` that holds a `.pth` file is used as a voice.
4. **Only if the GitHub repository is private:** Add-ons → Secrets → add `GITHUB_TOKEN` (a read-only token for this repository) and attach it to this notebook.
5. **Run all.** The first run takes about 10–15 minutes, mostly installing PyTorch.

When it is done, send back the GPU line and the two bench tables at the end.

In [ ]:
REPO = "https://github.com/ngmthaq/voice-clone.git"
BRANCH = "main"
SECONDS = 8  # seconds of audio per preset

WORK = "/kaggle/working"
APP = f"{WORK}/voice-clone"
VENV = f"{WORK}/venv312"
PY = f"{VENV}/bin/python"

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv

## 2. Get the app

`sounddevice` needs the PortAudio library to load, even though the test uses no audio devices.

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y libportaudio2 > /dev/null && echo "PortAudio installed"

In [ ]:
import os
import subprocess

url = REPO
try:
    from kaggle_secrets import UserSecretsClient

    url = REPO.replace("https://", f"https://x-access-token:{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    pass  # no secret: the repository must be public

if not os.path.isdir(APP):
    result = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, url, APP], capture_output=True, text=True
    )
    if result.returncode != 0:
        raise SystemExit(
            "Could not clone the repository. If it is private, add a GITHUB_TOKEN secret.\n"
            + result.stderr.replace(url, REPO)
        )
    # Keep the token out of the saved notebook output folder.
    subprocess.run(["git", "-C", APP, "remote", "set-url", "origin", REPO], check=True)

print(subprocess.run(["git", "-C", APP, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 3. Install Python 3.12 and the app

The app needs Python 3.12 exactly, so it gets its own environment next to Kaggle's Python.

In [ ]:
!pip install -q uv
!uv python install 3.12
!test -x {PY} || uv venv --python 3.12 {VENV}
!uv pip install --python {PY} -q torch==2.5.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124
!uv pip install --python {PY} -q -e {APP}

In [ ]:
!{PY} -c "import torch; from voice_clone.domain.engine.device import use_half_precision; d = torch.device('cuda'); print('PyTorch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0), '| compute capability', torch.cuda.get_device_capability(0), '| auto precision:', 'fp16' if use_half_precision(d, 'auto') else 'fp32')"

## 4. Add the voice and the base models

In [ ]:
import glob
import pathlib
import shutil

voices = pathlib.Path(APP, "voices")
models = sorted(glob.glob("/kaggle/input/**/*.pth", recursive=True))
if not models:
    raise SystemExit("No .pth voice found in /kaggle/input. Add your voice dataset (see the first cell).")
for model in models:
    source = pathlib.Path(model).parent
    target = voices / source.name
    shutil.copytree(source, target, dirs_exist_ok=True)
    print("Voice:", target.name, sorted(p.name for p in target.iterdir()))

In [ ]:
!cd {APP} && {PY} -c "from voice_clone.domain.engine import ensure_base_models; from voice_clone.domain.paths import models_dir; ensure_base_models(models_dir()); print('Base models ready')"

## 5. Speed test

**Load** = time to convert a block ÷ block length. Under 1.0 keeps up; the app recommends presets whose `p95` stays under 0.7. The last column shows where the time goes.

In [ ]:
!cd {APP} && {PY} -m voice_clone bench --precision fp32 --seconds {SECONDS} 2>&1 | grep -v -i "warn"

In [ ]:
!cd {APP} && {PY} -m voice_clone bench --precision fp16 --seconds {SECONDS} 2>&1 | grep -v -i "warn"

## What to send back

- The line from step 3 with the GPU name, compute capability and auto precision.
- Both bench tables (fp32 and fp16), with the **Recommended** lines.

Then run the notebook again with the other GPU (P100 / T4 x2). When you are done, you can delete the voice dataset from Kaggle.